# Practical Session 6: Linear Regression and Least Squares Modeling

This notebook treats linear regression as a least-squares problem and shows the full
workflow from matrix construction to scaling, residual analysis, and synthetic tests.


In [ ]:
# Import NumPy for matrix formulas and synthetic-data generation.
import numpy as np
# Import pandas for readable result tables.
import pandas as pd
# Import cvxpy for the optimization-based least-squares formulation.
import cvxpy as cp
# Import Matplotlib for residual and noise-level plots.
import matplotlib.pyplot as plt

# Print arrays compactly.
np.set_printoptions(precision=4, suppress=True)
# Select a solver for the optimization-based formulation.
SOLVER = "CLARABEL" if "CLARABEL" in cp.installed_solvers() else "SCS"
# Fix the seed for the synthetic experiment.
rng = np.random.default_rng(99)

# Store sample names for plot labels.
names = ["Moeller", "Kroos", "Reus", "Gomez", "Goetze"]
# Store the two raw input features.
X_raw = np.array(
    [
        [10.0, 0.1],
        [2.0, 0.7],
        [6.0, 0.6],
        [8.0, 0.1],
        [8.0, 0.4],
    ]
)
# Store the observed targets for the first four training samples.
y_train = np.array([11.0, 9.0, 12.0, 9.0])
# Split the data into training inputs and the new sample to be predicted.
X_train_raw = X_raw[:4]
x_goetze_raw = X_raw[4]


## Task 1: Matrix formulation

We add a bias column so the regression model takes the form `y_hat = X @ w` with one
intercept parameter and one coefficient per feature.


In [ ]:
def add_bias(X):
    # Prepend a column of ones so the intercept is absorbed into the parameter vector.
    # TODO: Add the bias column for the regression model.
    return ...


# Build the design matrix for the four labeled samples.
X_train = add_bias(X_train_raw)
# Build the design row for the prediction sample Goetze.
x_goetze = add_bias(x_goetze_raw.reshape(1, -1))

print("Design matrix shape:", X_train.shape)
print("Target vector shape:", y_train.shape)
print("Prediction uses y_hat = X @ w")


## Task 2: Solve the least-squares problem in two ways

The practical asks for both the normal-equation solution and an optimization-based
solution. We compute both and compare the resulting coefficients and predictions.


In [ ]:
# Solve the normal equations (X^T X) w = X^T y directly.
# TODO: Solve the normal equations for the regression coefficients.
w_normal = ...

# Create the optimization variable for the cvxpy formulation.
w_cvx = cp.Variable(X_train.shape[1])
# Minimize the squared residual norm.
# TODO: Build the optimization-based least-squares problem.
ls_problem = ...
ls_problem.solve(solver=SOLVER)
w_cvx_value = w_cvx.value

# Compute fitted values on the training set for both methods.
train_pred_normal = X_train @ w_normal
train_pred_cvx = X_train @ w_cvx_value
# Predict the target of Goetze with both fitted models.
goetze_pred_normal = float((x_goetze @ w_normal)[0])
goetze_pred_cvx = float((x_goetze @ w_cvx_value)[0])

# Compare the fitted parameter vectors side by side.
comparison_df = pd.DataFrame(
    {"normal_equations": w_normal, "cvxpy": w_cvx_value},
    index=["bias", "goals", "pcr"],
)
display(comparison_df)
print("Goetze prediction from normal equations:", goetze_pred_normal)
print("Goetze prediction from cvxpy:", goetze_pred_cvx)


## Task 3: Residual analysis

Residual diagnostics show how well the model matches the observed data and make the fit
quality visible.


In [ ]:
# Compute the residual vector y - y_hat.
residuals = y_train - train_pred_normal
# Compute standard quality metrics for the training set.
mse = np.mean(residuals**2)
rmse = np.sqrt(mse)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y_train - y_train.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

# Display the numerical quality measures.
metrics = pd.DataFrame([{"MSE": mse, "RMSE": rmse, "R2": r2}])
display(metrics)

# Plot observed versus predicted values and residuals.
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(y_train, train_pred_normal, s=80)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], "k--")
axes[0].set_xlabel("Observed target")
axes[0].set_ylabel("Predicted target")
axes[0].set_title("Predicted vs observed")
axes[0].grid(True, alpha=0.3)

axes[1].bar(names[:4], residuals)
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals by training sample")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Task 4: Feature scaling and numerical effects

Feature scaling can improve numerical conditioning while leaving predictions essentially
unchanged if the transformed model is used consistently.


In [ ]:
# Compute feature means and standard deviations on the training data.
feature_mean = X_train_raw.mean(axis=0)
feature_std = X_train_raw.std(axis=0, ddof=0)
# Standardize the training features to zero mean and unit variance.
# TODO: Standardize the training features before adding the bias column.
X_train_scaled = ...
# Apply the same transformation to the new sample.
x_goetze_scaled = add_bias(((x_goetze_raw - feature_mean) / feature_std).reshape(1, -1))

# Fit the scaled model by least squares.
w_scaled = np.linalg.lstsq(X_train_scaled, y_train, rcond=None)[0]
scaled_train_predictions = X_train_scaled @ w_scaled
scaled_goetze_prediction = float((x_goetze_scaled @ w_scaled)[0])

# Compare the conditioning before and after scaling.
condition_unscaled = np.linalg.cond(X_train.T @ X_train)
condition_scaled = np.linalg.cond(X_train_scaled.T @ X_train_scaled)

# Show the coefficient representations in both coordinate systems.
scaling_df = pd.DataFrame(
    {"unscaled_coefficients": w_normal, "scaled_coefficients": w_scaled},
    index=["bias", "feature_1", "feature_2"],
)
display(scaling_df)
print("Condition number before scaling:", condition_unscaled)
print("Condition number after scaling:", condition_scaled)
print("Goetze prediction after scaling:", scaled_goetze_prediction)


## Task 5: Polynomial feature extension

We augment the design matrix by nonlinear terms to study how a richer model behaves on
a very small data set.


In [ ]:
def polynomial_features(X):
    # Split the two original features to keep the expressions readable.
    x1 = X[:, [0]]
    x2 = X[:, [1]]
    # Build bias, linear, quadratic, and interaction terms.
    return np.hstack([np.ones((X.shape[0], 1)), x1, x2, x1**2, x2**2, x1 * x2])


# Create the extended feature matrices for training and prediction.
X_poly = polynomial_features(X_train_raw)
x_goetze_poly = polynomial_features(x_goetze_raw.reshape(1, -1))
# Fit the richer least-squares model.
w_poly = np.linalg.lstsq(X_poly, y_train, rcond=None)[0]
train_pred_poly = X_poly @ w_poly
goetze_pred_poly = float((x_goetze_poly @ w_poly)[0])

# Compute the training error of the polynomial model.
poly_mse = np.mean((y_train - train_pred_poly) ** 2)
print("Polynomial coefficients:", np.round(w_poly, 4))
print("Polynomial training MSE:", poly_mse)
print("Polynomial prediction for Goetze:", goetze_pred_poly)


## Task 6: Synthetic data experiment

The final experiment uses a larger synthetic data set with known ground truth so the
effect of observation noise on estimation quality becomes visible.


In [ ]:
# Define the true regression coefficients including the intercept.
true_w = np.array([1.5, -2.0, 0.8])
# Generate a moderately sized synthetic feature matrix.
n_samples = 150
X_syn = rng.normal(size=(n_samples, 2))
X_syn_design = add_bias(X_syn)

# Test several additive noise levels.
noise_levels = np.linspace(0.0, 1.0, 8)
synthetic_rows = []
for noise in noise_levels:
    # Add Gaussian observation noise with the current standard deviation.
    noise_vector = rng.normal(scale=noise, size=n_samples)
    y_syn = X_syn_design @ true_w + noise_vector
    # Estimate the coefficients from the noisy observations.
    estimated_w = np.linalg.lstsq(X_syn_design, y_syn, rcond=None)[0]
    predictions = X_syn_design @ estimated_w
    synthetic_rows.append(
        {
            "noise_level": noise,
            "coefficient_error": np.linalg.norm(estimated_w - true_w),
            "training_rmse": np.sqrt(np.mean((predictions - y_syn) ** 2)),
        }
    )

# Summarize the dependence on the noise level.
synthetic_df = pd.DataFrame(synthetic_rows)
display(synthetic_df)

# Plot how the training RMSE changes as the noise level increases.
plt.figure(figsize=(6, 4))
plt.plot(synthetic_df["noise_level"], synthetic_df["training_rmse"], marker="o")
plt.xlabel("Noise level")
plt.ylabel("Training RMSE")
plt.title("Prediction accuracy vs additive noise")
plt.grid(True, alpha=0.3)
plt.show()
